# Semana 04 — Manipulação de Arquivos e Modularização

---
## 1. Arquivos: CSV, JSON e Excel

### Exemplo 1 — Lendo o cadastro de funcionários (CSV)

In [2]:
import csv

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    funcionarios = list(leitor)

print(f"Total de funcionários: {len(funcionarios)}")
print(funcionarios[0])

Total de funcionários: 30
{'id_funcionario': '1', 'funcionario': 'Ana Souza', 'cargo': 'Atendente', 'setor': 'Comercial'}


### Exemplo 2 — Lendo os e-mails corporativos (JSON)

In [3]:
import json

with open("dataset/emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

print(emails[0])

{'id_funcionario': 1, 'funcionario': 'Ana Souza', 'email': 'ana.souza@grupoalfa.com.br'}


### Exemplo extra — Escrevendo um CSV
Para escrever, o Python usa `csv.DictWriter`


In [5]:
import csv

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

funcionarios_estoque = [f for f in funcionarios if f["setor"] == "Estoque"]

with open("dataset/funcionarios_estoque.csv", "w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=funcionarios[0].keys())
    escritor.writeheader()
    escritor.writerows(funcionarios_estoque)

print(f"{len(funcionarios_estoque)} funcionários do Estoque salvos em funcionarios_estoque.csv")

9 funcionários do Estoque salvos em funcionarios_estoque.csv


### Exemplo 3 — Lendo o banco de horas (Excel, múltiplas abas)
Para instalar openpyxl `%pip install -q openpyxl`

In [7]:
%pip install -q openpyxl

Note: you may need to restart the kernel to use updated packages.


In [8]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
print("Abas disponíveis:", planilha.sheetnames)

aba_ponto = planilha["banco_horas"]
print(f"Total de registros de ponto: {aba_ponto.max_row - 1}")  # -1 por causa do cabeçalho

for linha in aba_ponto.iter_rows(min_row=2, max_row=4, values_only=True):
    print(linha)

Abas disponíveis: ['cadastro_funcionario', 'banco_horas']
Total de registros de ponto: 100
(1, 'Atendente', 'Comercial', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 30), datetime.datetime(2023, 1, 2, 16, 25), 8)
(2, 'Motorista', 'Logística', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 30), datetime.datetime(2023, 1, 2, 19, 30), 8)
(3, 'Separador de Estoque', 'Estoque', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 9), datetime.datetime(2023, 1, 2, 16, 34), 8)


- `min_row=2` — comece a partir da linha 2 
- `max_row=4` — pare na linha 4
- `values_only=True` — devolva só os valores de cada célula


---
## 2. Datas e Expressões Regulares

### Datas com `datetime`
### Exemplo 1 — Calculando horas trabalhadas

In [9]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada = registro[4]
saida = registro[5]

horas_trabalhadas = (saida - entrada).seconds / 3600
print(f"Horas trabalhadas: {horas_trabalhadas:.2f}h")

Horas trabalhadas: 7.92h


### Exemplo 2 — Calculando o atraso em minutos

In [10]:
from openpyxl import load_workbook

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada_prevista_min = 8 * 60  # 08:00, em minutos
entrada_real = registro[4]
entrada_real_min = entrada_real.hour * 60 + entrada_real.minute

atraso_minutos = entrada_real_min - entrada_prevista_min
print(f"Atraso: {atraso_minutos} minutos")

Atraso: 30 minutos


### Expressões regulares com `re`
### Exemplo 1 — Validando um e-mail

In [11]:
import re

email = "ana.souza@grupoalfa.com.br"
padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"

if re.match(padrao, email):
    print("E-mail parece válido!")
else:
    print("E-mail inválido.")

E-mail parece válido!


### Exemplo extra — Validando um CPF

In [12]:
import re

cpf_valido = "123.456.789-00"
cpf_invalido = "123.456.789-0"  # faltou um dígito antes do traço

padrao_cpf = r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"

for cpf in [cpf_valido, cpf_invalido]:
    if re.match(padrao_cpf, cpf):
        print(f"{cpf}: CPF parece válido!")
    else:
        print(f"{cpf}: CPF inválido.")

123.456.789-00: CPF parece válido!
123.456.789-0: CPF inválido.


---
## 3. Funções: parâmetros, valores padrão e `return`

In [13]:
def dizer_ola():
    return "Olá!"

print(dizer_ola())

def dizer_ola_para(nome):
    return f"Olá, {nome}!"

print(dizer_ola_para("Ana"))

Olá!
Olá, Ana!


### Exemplo 1 — Função com retorno: horas trabalhadas

In [14]:
from openpyxl import load_workbook

def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

registro = None
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] == 1 and linha[3].date().isoformat() == "2023-01-02":
        registro = linha  # Ana Souza (id 1), 02/01/2023
        break

entrada = registro[4]
saida = registro[5]
print(f"{calcular_horas_trabalhadas(entrada, saida):.2f}h")

7.92h


### Exemplo 2 — Parâmetro com valor padrão: classificar pontualidade


In [15]:
def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado"

print(classificar_pontualidade(5))                   # dentro da tolerância padrão: Pontual
print(classificar_pontualidade(30))                  # acima da tolerância padrão: Atrasado
print(classificar_pontualidade(12, tolerancia=15))   # tolerância customizada: Pontual

Pontual
Atrasado
Pontual


---
## 4. Lambda e Módulos
### Exemplo 1 — Função lambda

In [16]:
calcular_atraso_lambda = lambda entrada_min, previsto_min=8*60: entrada_min - previsto_min

print(calcular_atraso_lambda(8*60 + 30))    # 30

30


### Exemplo 2 — Onde o lambda realmente brilha: dentro de `min()`/`sorted()`

In [17]:
import csv
from openpyxl import load_workbook

with open("dataset/cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))
nome_por_id = {int(f["id_funcionario"]): f["funcionario"] for f in funcionarios}

planilha = load_workbook("dataset/recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

ids_do_dia = [1, 2, 8]  # Ana Souza, Bruno Lima, Helena Martins
registros_do_dia = []
for linha in aba_ponto.iter_rows(min_row=2, values_only=True):
    if linha[0] in ids_do_dia and linha[3].date().isoformat() == "2023-01-02":
        registros_do_dia.append({"funcionario": nome_por_id[linha[0]], "entrada": linha[4]})

primeiro_a_chegar = min(registros_do_dia, key=lambda r: r["entrada"])
print(f"Primeiro a chegar: {primeiro_a_chegar['funcionario']}")

Primeiro a chegar: Helena Martins
